# Data Exploration for Industrial Fire Detection
This notebook loads the FIRMS active fire data and OSM industrial polygons for the Jamnagar region and visualizes them on an interactive map.

In [ ]:
import pandas as pd
import geopandas as gpd
import folium
import os
import glob

## 1. Load Data
Run the ingestion scripts (`python src/data/ingest_firms.py` and `python src/data/ingest_osm.py`) before running these cells!

In [ ]:
# Load OSM Polygons
osm_path = '../data/raw/osm_industrial_jamnagar.geojson'
if os.path.exists(osm_path):
    gdf_osm = gpd.read_file(osm_path)
    print(f"Loaded {len(gdf_osm)} industrial polygons.")
else:
    print("OSM data not found. Run ingest_osm.py first.")

# Load latest FIRMS CSV
firms_files = glob.glob('../data/raw/firms_*.csv')
if firms_files:
    latest_firms = max(firms_files, key=os.path.getctime)
    df_firms = pd.read_csv(latest_firms)
    print(f"Loaded {len(df_firms)} FIRMS hotspots from {latest_firms}.")
else:
    print("FIRMS data not found. Run ingest_firms.py first with your API key.")

## 2. Interactive Map Visualization

In [ ]:
# Create base map centered at Jamnagar
m = folium.Map(location=[22.47, 70.05], zoom_start=11)

# Add OSM Polygons (Industrial Zones)
if 'gdf_osm' in locals():
    folium.GeoJson(
        gdf_osm,
        name="Industrial Zones",
        style_function=lambda x: {'fillColor': 'blue', 'color': 'blue', 'weight': 1, 'fillOpacity': 0.3}
    ).add_to(m)

# Add FIRMS Hotspots
if 'df_firms' in locals():
    for idx, row in df_firms.iterrows():
        folium.CircleMarker(
            location=[row['latitude'], row['longitude']],
            radius=3,
            color='red',
            fill=True,
            fill_color='red',
            fill_opacity=0.7,
            popup=f"FRP: {row.get('frp', 'N/A')}<br>Acq Date: {row.get('acq_date', 'N/A')}"
        ).add_to(m)

folium.LayerControl().add_to(m)
m